# Scraping de books.toscrape.com avec Selenium

Ce notebook sert de **démonstration de Selenium** (nécessite Google Chrome). Le site étant statique, Selenium n'est pas nécessaire ici : [`Analyse_Livres.ipynb`](Analyse_Livres.ipynb) fait le même travail avec `requests` + Beautiful Soup, beaucoup plus vite, avec en plus validation des données, reprise après interruption et analyse.

Selenium reste utile pour les sites rendus en JavaScript ou qui demandent des interactions.

In [ ]:
import sys
import time

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

sys.path.append("../src")  # code réutilisable dans src/webscraping
from webscraping.fx import get_gbp_eur
from webscraping.storage import save_dataset

In [ ]:
import selenium
print(selenium.__version__)

4.44.0


In [ ]:
# Taux du jour (API publique, données de la BCE) ; valeur de secours si l'API est injoignable
taux = get_gbp_eur()
print(f"1 GBP = {taux.rate} EUR (source : {taux.source}, date : {taux.date})")


def ls_to_eu(ls):
    ls = str(ls).replace("£", "")  # retirer la devise
    return float(ls) * taux.rate  # convertit livre en euro

Étape 1 — parcourt les pages uniquement pour collecter les URLs

Étape 2 — visite chaque URL directement, les pages n'ont plus aucune importance

In [ ]:
debut = time.perf_counter()
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 3)  # attend 3 secondes si un élément est absent

book_urls = []  # collecte des URLs
data = []
errors = []

# Champs obligatoires : si l'un manque, la page est signalée en erreur
SELECTORS = {
    "categorie": (By.XPATH, "(//ul[@class='breadcrumb']//a)[last()]"),
    "titre":     (By.CSS_SELECTOR, "h1"),
    "prix":      (By.CSS_SELECTOR, ".price_color"),
    "stock":     (By.CSS_SELECTOR, ".instock.availability"),
}
# Champ optionnel : certains livres n'ont pas de description
DESCRIPTION = (By.CSS_SELECTOR, "#product_description + p")

try:
    driver.get("https://books.toscrape.com/")

    # ÉTAPE 1 : collecter toutes les URLs
    while True:  # boucle infinie, on sort manuellement avec break
        # Attend que les liens soient présents dans le DOM
        wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product_pod h3 a")))

        # Trouve tous les liens et extrait leur href, les ajoute à book_urls
        book_urls += [link.get_attribute("href") for link in driver.find_elements(By.CSS_SELECTOR, "article.product_pod h3 a")]

        try:  # cherche le bouton "next", clique dessus directement si trouvé
            wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "li.next a"))).click()
        except TimeoutException:
            # bouton "next" absent = dernière page, on sort de la boucle
            break

    print(f"{len(book_urls)} URLs collectées")

    # ÉTAPE 2 : extraire les données de chaque livre
    for url in book_urls:
        try:
            driver.get(url)  # navigue directement vers la page du livre
            row = {key: wait.until(EC.presence_of_element_located(sel)).text for key, sel in SELECTORS.items()}
            # find_elements ne lève pas d'erreur et n'attend pas : liste vide si pas de description
            description = driver.find_elements(*DESCRIPTION)
            row["description"] = description[0].text if description else None
            data.append(row)
        except TimeoutException:
            errors.append(url)
            print(f"Erreur sur {url}")
finally:
    driver.quit()  # ferme Chrome même si le notebook plante

dff = pd.DataFrame(data)
print(f"{len(dff)} lignes enregistrées, {len(errors)} erreurs")
print(f"Durée avec Selenium : {time.perf_counter() - debut:.0f} s")

2 livres n'ont pas de description : la colonne reste vide (NaN) au lieu de perdre la ligne, on obtient donc 1000 lignes.

In [ ]:
dff

In [ ]:
dff.info()

                                                                Nettoyage

In [ ]:
dff["prix"] = round(dff["prix"].apply(ls_to_eu), 2)

In [ ]:
# 1 : split : ["In stock ", "22 available)"]
# 2 : on garde le deuxime element
# 3 : split : ["22", "available)"]
# 4 : on garde le premiere element

dff["stock"] = dff["stock"].apply(lambda text: text.split("(")[1].split()[0])

In [ ]:
dff["stock"] = dff["stock"].astype(int) # converti en entier

In [ ]:
dff.head()

                                                               Exploration

In [ ]:
dff.describe()

In [ ]:

dff["categorie"].describe() # verifie le nb de categorie


In [ ]:

dff["categorie"].value_counts()


Note : "Default" et "Add a comment" sont de vraies catégories du site books.toscrape.com, pas des erreurs de scraping. Elles regroupent des livres sans vraie catégorie et faussent l'analyse par catégorie.

In [ ]:

dff["titre"].describe() # verifie les doublons


In [ ]:

dff[dff["titre"] == "The Star-Touched Queen"]


Note : ce sont deux fiches distinctes (prix et stock différents), pas un doublon à supprimer.

In [ ]:
dff_sorted = dff.sort_values(by="categorie")

dff_sorted.head()

In [ ]:
save_dataset(dff, "livres_selenium", "../data")